# Statistical analysis of the composition results


In [34]:
import json
from pathlib import Path
import pandas as pd
import numpy as np

REPO_ROOT = Path.cwd().parents[1]

In [68]:
# Load data
with open(REPO_ROOT / "results/composition/v1_phase12_normFalse_a4/scoring/summary.json") as f:
    summary = json.load(f)
  
# Metadata  
print(summary["model"], summary["layer"], summary["alpha"], summary["tau_value"])

# Per-composition entries
pairs = summary["pairs"]
print(pairs[0].keys)

# Extract per-composition data
rows = []
for p in pairs:
    if p.get("status") != "ok":
        continue
    rows.append({
        "trait_a": p["trait_a"], "trait_b": p["trait_b"],
        "cos": p["cos"], "regime": p["regime"],
        "comp_base": p["baseline"]["composition_mean"],
        "comp_steered": p["steered"]["composition_mean"],
        "coh_steered": p["steered"]["coherence_mean"],
        "delta_a_joint":  p["delta"]["trait_a_joint"],
        "delta_a_single": p["delta"]["trait_a_single"],
        "delta_b_joint":  p["delta"]["trait_b_joint"],
        "delta_b_single": p["delta"]["trait_b_single"],
        "delta_comp": p["delta"]["composition"],
        "delta_coh":  p["delta"]["coherence"],
    })
    
# Convert to pd dataframe
df = pd.DataFrame(rows)
df["regime"].value_counts()

meta-llama/Llama-3.1-8B-Instruct 17 4.0 0.8900726269483555
<built-in method keys of dict object at 0x12c958b40>


regime
mixed          19
emergent        6
dominant        5
additive        3
suppressive     3
Name: count, dtype: int64

In [69]:
# Add semantic similarity
with open(REPO_ROOT / "results/semantic_similarity.json") as f:
    sem = json.load(f)
    
sem_lookup = {
    tuple(sorted([p["trait_a"], p["trait_b"]])): p["sem_sim"]
    for p in sem["pairs"]
}

df["sem_sim"] = df.apply(
    lambda r: sem_lookup[tuple(sorted([r["trait_a"], r["trait_b"]]))],
    axis=1,
)

In [70]:
# Look at dataframe
df.head(n=40)

,trait_a,trait_b,cos,regime,comp_base,comp_steered,coh_steered,delta_a_joint,delta_a_single,delta_b_joint,delta_b_single,delta_comp,delta_coh,sem_sim
0,apathetic,confidence,0.0143,dominant,32.70,54.85,55.56,57.13,32.65,-12.84,20.08,22.15,-42.23,0.240104
1,apathetic,evil,0.2578,mixed,4.79,54.88,28.75,73.73,36.76,26.44,45.90,50.09,-68.62,0.321475
2,apathetic,formality,0.0150,mixed,44.56,78.29,80.87,64.24,21.48,3.22,7.85,33.73,-16.61,0.246348
3,apathetic,hallucinating,-0.1642,emergent,10.37,56.44,59.95,43.39,23.64,48.76,28.16,46.07,-32.51,0.198113
4,apathetic,humorous,0.1923,mixed,1.25,65.28,27.09,78.35,22.34,49.71,75.80,64.03,-69.25,0.232589
5,apathetic,impolite,0.6948,mixed,0.93,76.79,23.47,80.17,28.00,71.56,66.36,75.86,-72.93,0.469881
6,apathetic,power_seeking,0.0063,additive,10.27,25.73,81.73,40.38,34.56,-9.46,-8.23,15.46,-15.53,0.291267
7,apathetic,sycophantic,0.0503,mixed,3.96,51.37,43.79,62.99,34.05,31.84,69.07,47.42,-53.52,0.344080
8,confidence,evil,0.2294,dominant,28.39,53.26,34.45,-14.58,24.39,64.31,37.09,24.87,-62.72,0.209184
9,confidence,formality,0.2262,mixed,75.96,82.50,78.46,10.44,17.56,2.64,5.60,6.54,-18.95,0.398346


### Cross-tab for cosine v. regime

We perform a descriptive analysis to see whether simple cross-tabulation predicts a regime using cosine similarity (Gram) matrix $G$.

In [71]:
# Add column with absolute value of cosine similarities
df['cosine_abs'] = df['cos'].abs()

# Bin cosine with the same strata of the previous analysis
df['cos_bin'] = pd.cut(
    df['cosine_abs'],
    bins=[0, 0.15, 0.30, 0.40],
    labels=['low', 'moderate', 'high']
)

df.head()

,trait_a,trait_b,cos,regime,comp_base,comp_steered,coh_steered,delta_a_joint,delta_a_single,delta_b_joint,delta_b_single,delta_comp,delta_coh,sem_sim,cosine_abs,cos_bin
0,apathetic,confidence,0.0143,dominant,32.70,54.85,55.56,57.13,32.65,-12.84,20.08,22.15,-42.23,0.240104,0.0143,low
1,apathetic,evil,0.2578,mixed,4.79,54.88,28.75,73.73,36.76,26.44,45.90,50.09,-68.62,0.321475,0.2578,moderate
2,apathetic,formality,0.0150,mixed,44.56,78.29,80.87,64.24,21.48,3.22,7.85,33.73,-16.61,0.246348,0.0150,low
3,apathetic,hallucinating,-0.1642,emergent,10.37,56.44,59.95,43.39,23.64,48.76,28.16,46.07,-32.51,0.198113,0.1642,moderate
4,apathetic,humorous,0.1923,mixed,1.25,65.28,27.09,78.35,22.34,49.71,75.80,64.03,-69.25,0.232589,0.1923,moderate


In [72]:
# Cross-tab
pd.crosstab(df['cos_bin'], df['regime'])

regime,additive,dominant,emergent,mixed,suppressive
cos_bin,,,,,
low,2,3,2,4,1
moderate,1,2,3,7,1
high,0,0,1,4,0


In [73]:
# Statistical chi-square test on cross-tab
from scipy.stats import chi2_contingency
chi2, p, dof, expected = chi2_contingency(
    pd.crosstab(df['cos_bin'], df['regime'])
)
print(f"\n chi2={chi2}, \n p={p}, \n dof={dof}, \n expected={expected}")


 chi2=4.681984126984126, 
 p=0.790962147758917, 
 dof=8, 
 expected=[[1.16129032 1.93548387 2.32258065 5.80645161 0.77419355]
 [1.35483871 2.25806452 2.70967742 6.77419355 0.90322581]
 [0.48387097 0.80645161 0.96774194 2.41935484 0.32258065]]


### Correlation between cosine and semantic similarities

In [74]:
corr = df[["cosine_abs", "sem_sim"]].corr().iloc[0, 1]
print(f"Pearson r(cosine_abs, sem_sim) = {corr:+.3f}")
print(df[["cosine_abs", "sem_sim"]].describe().round(3))

Pearson r(cosine_abs, sem_sim) = +0.528
       cosine_abs  sem_sim
count      36.000   36.000
mean        0.232    0.274
std         0.159    0.073
min         0.006    0.185
25%         0.105    0.222
50%         0.228    0.245
75%         0.316    0.322
max         0.695    0.470


## Logistic regression

We perform multiple logistic regressions to see whether the cosine similarity $|G_{i,j}|$ for composition of behavior $i$ and behavior $j$ is a predictor for the composition regime.
1. First, we perform a binary regression using only absolute cosine similarities, to predict additive v. non-additive;
2. We repeat step 1 using signed cosine to see whether sign matters;
3. We try a multinomial logistic regression to predict all regimes;
4. We add semantic similarity as a confound control and repeat all the above steps.

In [75]:
# Save experiments results
results = []

In [76]:
# Add binary variable for additive v. non-additive
df['is_additive'] = (df['regime'] == 'additive').astype(int)
df.head()

,trait_a,trait_b,cos,regime,comp_base,comp_steered,coh_steered,delta_a_joint,delta_a_single,delta_b_joint,delta_b_single,delta_comp,delta_coh,sem_sim,cosine_abs,cos_bin,is_additive
0,apathetic,confidence,0.0143,dominant,32.70,54.85,55.56,57.13,32.65,-12.84,20.08,22.15,-42.23,0.240104,0.0143,low,0
1,apathetic,evil,0.2578,mixed,4.79,54.88,28.75,73.73,36.76,26.44,45.90,50.09,-68.62,0.321475,0.2578,moderate,0
2,apathetic,formality,0.0150,mixed,44.56,78.29,80.87,64.24,21.48,3.22,7.85,33.73,-16.61,0.246348,0.0150,low,0
3,apathetic,hallucinating,-0.1642,emergent,10.37,56.44,59.95,43.39,23.64,48.76,28.16,46.07,-32.51,0.198113,0.1642,moderate,0
4,apathetic,humorous,0.1923,mixed,1.25,65.28,27.09,78.35,22.34,49.71,75.80,64.03,-69.25,0.232589,0.1923,moderate,0


In [77]:
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.model_selection import cross_val_predict, StratifiedKFold, LeaveOneOut

# Single rng instantiated
rng = np.random.default_rng(0)
N_BOOT  = 2000
N_PERM  = 10000

In [78]:
def bootstrap_perm(model, X, y, exp_auc):

    def fit_auc(X_tr, y_tr, X_ev, y_ev):
        m = clone(model).fit(X_tr, y_tr)
        return roc_auc_score(y_ev, m.predict_proba(X_ev)[:, 1])

    pos_idx = np.where(y == 1)[0]
    neg_idx = np.where(y == 0)[0]
    n_obs   = len(y)

    # --- LOO slope stability ---
    # For each of the 36 leave-one-out folds: fit on n-1 points, extract the
    # slope coefficient (coef_[0, :]), track sign and range.
    # Also count folds where the training set has < 2 positives — with 3
    # positives total and LOO, this only fires when a positive is held out AND
    # there are only 2 remaining. Expected count: 0 (we have 3 positives so
    # the worst case is 2 in training), but the check is a useful sanity gate
    # for future experiments with fewer positives.
    loo_slopes        = []
    loo_degen_folds   = 0
    for i in range(n_obs):
        mask   = np.ones(n_obs, dtype=bool)
        mask[i] = False
        y_tr   = y[mask]
        if (y_tr == 1).sum() < 2:
            loo_degen_folds += 1
            continue
        try:
            m = clone(model).fit(X[mask], y_tr)
            loo_slopes.append(m.coef_[0].tolist())    # one value per feature
        except Exception:
            pass

    loo_slopes_arr = np.array(loo_slopes)              # shape: (n_valid_folds, n_features)
    print(f"LOO folds with < 2 positives in training: {loo_degen_folds}")
    for feat_idx in range(loo_slopes_arr.shape[1]):
        col = loo_slopes_arr[:, feat_idx]
        print(f"  feature {feat_idx}: slope min={col.min():+.3f}  "
              f"max={col.max():+.3f}  "
              f"n_negative={int((col < 0).sum())}/{len(col)}")

    # --- Bootstrap 95% CI (stratified, evaluate on resample) ---
    boot_aucs = []
    for _ in range(N_BOOT):
        pi  = rng.choice(pos_idx, size=len(pos_idx), replace=True)
        ni  = rng.choice(neg_idx, size=len(neg_idx), replace=True)
        idx = np.concatenate([pi, ni])
        if len(np.unique(y[idx])) < 2:
            continue
        try:
            boot_aucs.append(fit_auc(X[idx], y[idx], X[idx], y[idx]))
        except ValueError:
            pass

    boot_aucs = np.array(boot_aucs)
    ci_lo, ci_hi = np.percentile(boot_aucs, [2.5, 97.5])
    print(f"AUC = {exp_auc:.3f}  95% CI [{ci_lo:.3f}, {ci_hi:.3f}]  "
          f"(n_boot={len(boot_aucs)}  shape: min={boot_aucs.min():.3f} "
          f"median={np.median(boot_aucs):.3f} max={boot_aucs.max():.3f})")

    # --- Permutation p-value (one-sided) ---
    perm_aucs = []
    for _ in range(N_PERM):
        y_perm = rng.permutation(y)
        perm_aucs.append(fit_auc(X, y_perm, X, y_perm))

    perm_aucs = np.array(perm_aucs)
    p_value   = (np.sum(perm_aucs >= exp_auc) + 1) / (N_PERM + 1)
    print(f"permutation p (one-sided) = {p_value:.4f}")

    return {
        "auc_insample":           exp_auc,
        "ci_lo":                  float(ci_lo),
        "ci_hi":                  float(ci_hi),
        "n_boot":                 len(boot_aucs),
        "boot_aucs":              boot_aucs.tolist(),      # full distribution for figures
        "perm_p":                 float(p_value),
        "perm_auc_mean":          float(perm_aucs.mean()),
        "loo_slope_min":          float(loo_slopes_arr[:, 0].min()),
        "loo_slope_max":          float(loo_slopes_arr[:, 0].max()),
        "loo_slope_n_neg":        int((loo_slopes_arr[:, 0] < 0).sum()),
        "loo_slope_n_folds":      len(loo_slopes),
        "loo_degen_folds":        loo_degen_folds,
    }

In [79]:
scaler = StandardScaler()
binary_model = LogisticRegression(penalty=None)
multi_model = LogisticRegression(penalty=None, multi_class='multinomial')

### Experiment 1

In [80]:
# Load exp 1 data
X_abs   = df[['cosine_abs']].to_numpy()
X_abs_s = scaler.fit_transform(X_abs)
y_add   = df["is_additive"].to_numpy()

# Fit on all data for in-sample AUC + coefficient sign
exp1_results = binary_model.fit(X_abs_s, y_add)
y_proba_in   = exp1_results.predict_proba(X_abs_s)[:, 1]
exp1_auc     = roc_auc_score(y_add, y_proba_in)

# Leave-one-out: fit on n-1, predict held-out probability, pool 36 probabilities, single AUC
loo            = LeaveOneOut()
y_proba_loo    = cross_val_predict(binary_model, X_abs_s, y_add, cv=loo, method="predict_proba")[:, 1]
exp1_auc_loo   = roc_auc_score(y_add, y_proba_loo)

print(f"in-sample AUC      = {exp1_auc:.3f}")
print(f"LOO (pooled) AUC   = {exp1_auc_loo:.3f}")
print(f"slope on cosine_abs (scaled) = {exp1_results.coef_[0,0]:+.3f}  "
      f"(negative => high |cos| reduces P(additive) — the geometric prediction)")

in-sample AUC      = 0.828
LOO (pooled) AUC   = 0.606
slope on cosine_abs (scaled) = -1.865  (negative => high |cos| reduces P(additive) — the geometric prediction)


In [81]:
# Run CI bootstrap and p-value permutation
stats = bootstrap_perm(binary_model, X_abs_s, y_add, exp1_auc)
results.append({
    "experiment":  "1_abs_cos",
    "outcome":     "is_additive",
    "predictors":  "cosine_abs",
    "n_features":  1,
    "auc_loo":     exp1_auc_loo,
    "slope_abs_cos": float(exp1_results.coef_[0, 0]),
    "slope_both_antisocial": None,         # not in this model
    "slope_signed_cos":      None,
    **stats,
})

LOO folds with < 2 positives in training: 0
  feature 0: slope min=-4.232  max=-1.163  n_negative=36/36
AUC = 0.828  95% CI [0.606, 1.000]  (n_boot=2000  shape: min=0.424 median=0.838 max=1.000)
permutation p (one-sided) = 0.0648


### Experiment 2

In [82]:
# Load exp 2 data
X_sem   = df[['sem_sim']].to_numpy()
X_sem_s = scaler.fit_transform(X_sem)
y_add   = df["is_additive"].to_numpy()

# Fit on all data for in-sample AUC + coefficient sign
exp2_results = binary_model.fit(X_sem_s, y_add)
y_proba_in   = exp2_results.predict_proba(X_sem_s)[:, 1]
exp2_auc     = roc_auc_score(y_add, y_proba_in)

# Leave-one-out: fit on n-1, predict held-out probability, pool 36 probabilities, single AUC
loo            = LeaveOneOut()
y_proba_loo    = cross_val_predict(binary_model, X_sem_s, y_add, cv=loo, method="predict_proba")[:, 1]
exp2_auc_loo   = roc_auc_score(y_add, y_proba_loo)

print(f"in-sample AUC      = {exp2_auc:.3f}")
print(f"LOO (pooled) AUC   = {exp2_auc_loo:.3f}")
print(f"slope on semantic_similarity (scaled) = {exp2_results.coef_[0,0]:+.3f}  "
      f"(negative => high |sem_sim| reduces P(additive) — the geometric prediction)")

in-sample AUC      = 0.646
LOO (pooled) AUC   = 0.253
slope on semantic_similarity (scaled) = -0.725  (negative => high |sem_sim| reduces P(additive) — the geometric prediction)


In [83]:
# Run CI bootstrap and p-value permutation
stats = bootstrap_perm(binary_model, X_sem_s, y_add, exp2_auc)
results.append({
    "experiment":  "2_sem_sim",
    "outcome":     "is_additive",
    "predictors":  "semantic_similarity",
    "n_features":  1,
    "auc_loo":     exp2_auc_loo,
    "slope_sem_sim": float(exp2_results.coef_[0, 0]),
    "slope_both_antisocial": None,         # not in this model
    "slope_signed_cos":      None,
    **stats,
})

LOO folds with < 2 positives in training: 0
  feature 0: slope min=-2.078  max=-0.219  n_negative=36/36
AUC = 0.646  95% CI [0.444, 0.939]  (n_boot=2000  shape: min=0.354 median=0.667 max=1.000)
permutation p (one-sided) = 0.4413


### Experiment 3

In [84]:
X_sgn   = df[['cos']].to_numpy()
X_sgn_s = scaler.fit_transform(X_sgn)
y_add   = df["is_additive"].to_numpy()

exp3_results = binary_model.fit(X_sgn_s, y_add)
y_proba_in_3 = exp3_results.predict_proba(X_sgn_s)[:, 1]
exp3_auc     = roc_auc_score(y_add, y_proba_in_3)

loo           = LeaveOneOut()
y_proba_loo_3 = cross_val_predict(binary_model, X_sgn_s, y_add, cv=loo, method="predict_proba")[:, 1]
exp3_auc_loo  = roc_auc_score(y_add, y_proba_loo_3)

print(f"in-sample AUC      = {exp3_auc:.3f}")
print(f"LOO (pooled) AUC   = {exp3_auc_loo:.3f}")
print(f"slope on signed_cos (scaled) = {exp3_results.coef_[0,0]:+.3f}  "
      f"(negative => high signed_cos reduces P(additive))")

in-sample AUC      = 0.677
LOO (pooled) AUC   = 0.121
slope on signed_cos (scaled) = -0.330  (negative => high signed_cos reduces P(additive))


In [85]:
stats = bootstrap_perm(binary_model, X_sgn_s, y_add, exp3_auc)
results.append({
    "experiment":  "3_signed_cos",
    "outcome":     "is_additive",
    "predictors":  "signed_cos",
    "n_features":  1,
    "auc_loo":     exp3_auc_loo,
    "slope_abs_cos":         None,
    "slope_both_antisocial": None,
    "slope_signed_cos":      float(exp3_results.coef_[0, 0]),
    **stats,
})

LOO folds with < 2 positives in training: 0
  feature 0: slope min=-0.604  max=-0.165  n_negative=36/36
AUC = 0.677  95% CI [0.455, 0.848]  (n_boot=2000  shape: min=0.333 median=0.677 max=0.970)
permutation p (one-sided) = 0.3457


### Experiment 4

In [86]:
def bootstrap_perm_multi(model, X, y, exp_auc):
    classes = sorted(np.unique(y))
    n_obs   = len(y)
    n_feat  = X.shape[1]

    def fit_auc_multi(X_tr, y_tr, X_ev, y_ev):
        if len(np.unique(y_tr)) < len(classes):
            return None
        m = clone(model).fit(X_tr, y_tr)
        if len(m.classes_) < len(classes):
            return None
        return roc_auc_score(y_ev, m.predict_proba(X_ev), multi_class='ovr', average='macro')

    loo_slopes      = []
    loo_degen_folds = 0
    for i in range(n_obs):
        mask = np.ones(n_obs, dtype=bool)
        mask[i] = False
        y_tr = y[mask]
        if len(np.unique(y_tr)) < len(classes):
            loo_degen_folds += 1
            continue
        try:
            m = clone(model).fit(X[mask], y_tr)
            loo_slopes.append(np.abs(m.coef_).mean(axis=0).tolist())
        except Exception:
            pass

    loo_slopes_arr = np.array(loo_slopes)
    print(f"LOO folds with missing class in training: {loo_degen_folds}")
    for feat_idx in range(n_feat):
        col = loo_slopes_arr[:, feat_idx]
        print(f"  feature {feat_idx}: mean|slope| min={col.min():.3f}  max={col.max():.3f}")

    boot_aucs = []
    for _ in range(N_BOOT):
        idx = rng.choice(n_obs, size=n_obs, replace=True)
        try:
            v = fit_auc_multi(X[idx], y[idx], X[idx], y[idx])
            if v is not None:
                boot_aucs.append(v)
        except Exception:
            pass

    boot_aucs = np.array(boot_aucs)
    ci_lo, ci_hi = np.percentile(boot_aucs, [2.5, 97.5])
    print(f"AUC = {exp_auc:.3f}  95% CI [{ci_lo:.3f}, {ci_hi:.3f}]  "
          f"(n_boot={len(boot_aucs)}  shape: min={boot_aucs.min():.3f} "
          f"median={np.median(boot_aucs):.3f} max={boot_aucs.max():.3f})")

    perm_aucs = []
    for _ in range(N_PERM):
        y_perm = rng.permutation(y)
        try:
            v = fit_auc_multi(X, y_perm, X, y_perm)
            if v is not None:
                perm_aucs.append(v)
        except Exception:
            pass

    perm_aucs = np.array(perm_aucs)
    p_value   = (np.sum(perm_aucs >= exp_auc) + 1) / (N_PERM + 1)
    print(f"permutation p (one-sided) = {p_value:.4f}")

    return {
        "auc_insample":    exp_auc,
        "ci_lo":           float(ci_lo),
        "ci_hi":           float(ci_hi),
        "n_boot":          len(boot_aucs),
        "boot_aucs":       boot_aucs.tolist(),
        "perm_p":          float(p_value),
        "perm_auc_mean":   float(perm_aucs.mean()),
        "loo_degen_folds": loo_degen_folds,
        "loo_n_folds":     len(loo_slopes),
    }

In [87]:
y_multi   = df["regime"].to_numpy()
X_abs_s_4 = scaler.fit_transform(df[['cosine_abs']].to_numpy())

exp4_results = multi_model.fit(X_abs_s_4, y_multi)
probas_in_4  = exp4_results.predict_proba(X_abs_s_4)
exp4_auc     = roc_auc_score(y_multi, probas_in_4, multi_class='ovr', average='macro')

loo           = LeaveOneOut()
y_proba_loo_4 = cross_val_predict(multi_model, X_abs_s_4, y_multi, cv=loo, method="predict_proba")
exp4_auc_loo  = roc_auc_score(y_multi, y_proba_loo_4, multi_class='ovr', average='macro')

print(f"in-sample AUC (macro OVR) = {exp4_auc:.3f}")
print(f"LOO (macro OVR)           = {exp4_auc_loo:.3f}")
for i, c in enumerate(exp4_results.classes_):
    print(f"  class {c}: coef = {exp4_results.coef_[i, 0]:+.3f}")

in-sample AUC (macro OVR) = 0.698
LOO (macro OVR)           = 0.434
  class additive: coef = -1.463
  class dominant: coef = -0.742
  class emergent: coef = +0.358
  class mixed: coef = +0.914
  class suppressive: coef = +0.933


/Users/federicoscaffidimuta/Desktop/Third year/ML project/steering-vector-composition/venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/federicoscaffidimuta/Desktop/Third year/ML project/steering-vector-composition/venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/federicoscaffidimuta/Desktop/Third year/ML project/steering-vector-composition/venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will a

In [88]:
stats = bootstrap_perm_multi(multi_model, X_abs_s_4, y_multi, exp4_auc)
results.append({
    "experiment":  "4_multi_abs_cos",
    "outcome":     "regime",
    "predictors":  "cosine_abs",
    "n_features":  1,
    "auc_loo":     exp4_auc_loo,
    **stats,
})

LOO folds with missing class in training: 0
  feature 0: mean|slope| min=0.714  max=1.494


/Users/federicoscaffidimuta/Desktop/Third year/ML project/steering-vector-composition/venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/federicoscaffidimuta/Desktop/Third year/ML project/steering-vector-composition/venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/federicoscaffidimuta/Desktop/Third year/ML project/steering-vector-composition/venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will a

AUC = 0.698  95% CI [0.620, 0.854]  (n_boot=1832  shape: min=0.540 median=0.739 max=0.945)


/Users/federicoscaffidimuta/Desktop/Third year/ML project/steering-vector-composition/venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/federicoscaffidimuta/Desktop/Third year/ML project/steering-vector-composition/venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/federicoscaffidimuta/Desktop/Third year/ML project/steering-vector-composition/venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will a

permutation p (one-sided) = 0.0603


/Users/federicoscaffidimuta/Desktop/Third year/ML project/steering-vector-composition/venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/federicoscaffidimuta/Desktop/Third year/ML project/steering-vector-composition/venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/federicoscaffidimuta/Desktop/Third year/ML project/steering-vector-composition/venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will a

### Experiment 5

In [89]:
X_both_5   = df[['cosine_abs', 'sem_sim']].to_numpy()
X_both_s_5 = scaler.fit_transform(X_both_5)

exp5_results = binary_model.fit(X_both_s_5, y_add)
y_proba_in_5 = exp5_results.predict_proba(X_both_s_5)[:, 1]
exp5_auc     = roc_auc_score(y_add, y_proba_in_5)

loo           = LeaveOneOut()
y_proba_loo_5 = cross_val_predict(binary_model, X_both_s_5, y_add, cv=loo, method="predict_proba")[:, 1]
exp5_auc_loo  = roc_auc_score(y_add, y_proba_loo_5)

print(f"in-sample AUC      = {exp5_auc:.3f}")
print(f"LOO (pooled) AUC   = {exp5_auc_loo:.3f}")
print(f"slope on cosine_abs (scaled) = {exp5_results.coef_[0,0]:+.3f}")
print(f"slope on sem_sim   (scaled)  = {exp5_results.coef_[0,1]:+.3f}")

in-sample AUC      = 0.818
LOO (pooled) AUC   = 0.475
slope on cosine_abs (scaled) = -1.830
slope on sem_sim   (scaled)  = -0.400


In [90]:
stats = bootstrap_perm(binary_model, X_both_s_5, y_add, exp5_auc)
results.append({
    "experiment":  "5_abs_cos_sem",
    "outcome":     "is_additive",
    "predictors":  "cosine_abs + sem_sim",
    "n_features":  2,
    "auc_loo":     exp5_auc_loo,
    "slope_abs_cos":         float(exp5_results.coef_[0, 0]),
    "slope_both_antisocial": None,
    "slope_signed_cos":      None,
    "slope_sem_sim":         float(exp5_results.coef_[0, 1]),
    **stats,
})

LOO folds with < 2 positives in training: 0
  feature 0: slope min=-4.715  max=-0.817  n_negative=36/36
  feature 1: slope min=-1.764  max=+0.280  n_negative=35/36
AUC = 0.818  95% CI [0.657, 1.000]  (n_boot=2000  shape: min=0.485 median=0.848 max=1.000)
permutation p (one-sided) = 0.2122


### Experiment 6

In [91]:
X_both_6   = df[['cos', 'sem_sim']].to_numpy()
X_both_s_6 = scaler.fit_transform(X_both_6)

exp6_results = binary_model.fit(X_both_s_6, y_add)
y_proba_in_6 = exp6_results.predict_proba(X_both_s_6)[:, 1]
exp6_auc     = roc_auc_score(y_add, y_proba_in_6)

loo           = LeaveOneOut()
y_proba_loo_6 = cross_val_predict(binary_model, X_both_s_6, y_add, cv=loo, method="predict_proba")[:, 1]
exp6_auc_loo  = roc_auc_score(y_add, y_proba_loo_6)

print(f"in-sample AUC      = {exp6_auc:.3f}")
print(f"LOO (pooled) AUC   = {exp6_auc_loo:.3f}")
print(f"slope on signed_cos (scaled) = {exp6_results.coef_[0,0]:+.3f}")
print(f"slope on sem_sim   (scaled)  = {exp6_results.coef_[0,1]:+.3f}")

in-sample AUC      = 0.667
LOO (pooled) AUC   = 0.273
slope on signed_cos (scaled) = -0.381
slope on sem_sim   (scaled)  = -0.701


In [92]:
stats = bootstrap_perm(binary_model, X_both_s_6, y_add, exp6_auc)
results.append({
    "experiment":  "6_signed_cos_sem",
    "outcome":     "is_additive",
    "predictors":  "signed_cos + sem_sim",
    "n_features":  2,
    "auc_loo":     exp6_auc_loo,
    "slope_abs_cos":         None,
    "slope_both_antisocial": None,
    "slope_signed_cos":      float(exp6_results.coef_[0, 0]),
    "slope_sem_sim":         float(exp6_results.coef_[0, 1]),
    **stats,
})

LOO folds with < 2 positives in training: 0
  feature 0: slope min=-0.633  max=+0.066  n_negative=35/36
  feature 1: slope min=-2.118  max=-0.208  n_negative=36/36
AUC = 0.667  95% CI [0.515, 0.970]  (n_boot=2000  shape: min=0.394 median=0.727 max=1.000)
permutation p (one-sided) = 0.6423


### Experiment 7

In [93]:
X_both_7   = df[['cosine_abs', 'sem_sim']].to_numpy()
X_both_s_7 = scaler.fit_transform(X_both_7)

exp7_results = multi_model.fit(X_both_s_7, y_multi)
probas_in_7  = exp7_results.predict_proba(X_both_s_7)
exp7_auc     = roc_auc_score(y_multi, probas_in_7, multi_class='ovr', average='macro')

loo           = LeaveOneOut()
y_proba_loo_7 = cross_val_predict(multi_model, X_both_s_7, y_multi, cv=loo, method="predict_proba")
exp7_auc_loo  = roc_auc_score(y_multi, y_proba_loo_7, multi_class='ovr', average='macro')

print(f"in-sample AUC (macro OVR) = {exp7_auc:.3f}")
print(f"LOO (macro OVR)           = {exp7_auc_loo:.3f}")
for i, c in enumerate(exp7_results.classes_):
    print(f"  class {c}: coef_abs={exp7_results.coef_[i,0]:+.3f}  coef_sem={exp7_results.coef_[i,1]:+.3f}")

in-sample AUC (macro OVR) = 0.714
LOO (macro OVR)           = 0.417
  class additive: coef_abs=-1.399  coef_sem=-0.342
  class dominant: coef_abs=-0.662  coef_sem=-0.146
  class emergent: coef_abs=+0.492  coef_sem=-0.254
  class mixed: coef_abs=+0.893  coef_sem=+0.177
  class suppressive: coef_abs=+0.675  coef_sem=+0.566


/Users/federicoscaffidimuta/Desktop/Third year/ML project/steering-vector-composition/venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/federicoscaffidimuta/Desktop/Third year/ML project/steering-vector-composition/venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/federicoscaffidimuta/Desktop/Third year/ML project/steering-vector-composition/venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will a

In [94]:
stats = bootstrap_perm_multi(multi_model, X_both_s_7, y_multi, exp7_auc)
results.append({
    "experiment":  "7_multi_abs_sem",
    "outcome":     "regime",
    "predictors":  "cosine_abs + sem_sim",
    "n_features":  2,
    "auc_loo":     exp7_auc_loo,
    **stats,
})

LOO folds with missing class in training: 0
  feature 0: mean|slope| min=0.586  max=1.564
  feature 1: mean|slope| min=0.191  max=0.576


/Users/federicoscaffidimuta/Desktop/Third year/ML project/steering-vector-composition/venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/federicoscaffidimuta/Desktop/Third year/ML project/steering-vector-composition/venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/federicoscaffidimuta/Desktop/Third year/ML project/steering-vector-composition/venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will a

AUC = 0.714  95% CI [0.656, 0.915]  (n_boot=1820  shape: min=0.516 median=0.790 max=0.982)


/Users/federicoscaffidimuta/Desktop/Third year/ML project/steering-vector-composition/venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/federicoscaffidimuta/Desktop/Third year/ML project/steering-vector-composition/venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/federicoscaffidimuta/Desktop/Third year/ML project/steering-vector-composition/venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will a

permutation p (one-sided) = 0.2748


/Users/federicoscaffidimuta/Desktop/Third year/ML project/steering-vector-composition/venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/federicoscaffidimuta/Desktop/Third year/ML project/steering-vector-composition/venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/federicoscaffidimuta/Desktop/Third year/ML project/steering-vector-composition/venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will a

In [95]:
results_df = pd.DataFrame(results)
results_df

,experiment,outcome,predictors,n_features,auc_loo,slope_abs_cos,slope_both_antisocial,slope_signed_cos,auc_insample,ci_lo,...,boot_aucs,perm_p,perm_auc_mean,loo_slope_min,loo_slope_max,loo_slope_n_neg,loo_slope_n_folds,loo_degen_folds,slope_sem_sim,loo_n_folds
0,1_abs_cos,is_additive,cosine_abs,1,0.606061,-1.864617,NaN,NaN,0.828283,0.606061,...,"[0.6565656565656566, 0.898989898989899, 0.6666...",0.064794,0.640015,-4.232471,-1.163039,36.0,36.0,0,NaN,NaN
1,2_sem_sim,is_additive,semantic_similarity,1,0.252525,NaN,NaN,NaN,0.646465,0.444444,...,"[0.6464646464646465, 0.5454545454545454, 0.676...",0.441256,0.636460,-2.077805,-0.219103,36.0,36.0,0,-0.724995,NaN
2,3_signed_cos,is_additive,signed_cos,1,0.121212,NaN,NaN,-0.330049,0.676768,0.454545,...,"[0.5757575757575757, 0.7070707070707071, 0.575...",0.345665,0.640323,-0.603698,-0.165387,36.0,36.0,0,NaN,NaN
3,4_multi_abs_cos,regime,cosine_abs,1,0.434175,NaN,NaN,NaN,0.698283,0.619892,...,"[0.736964384971975, 0.7549895420405477, 0.8194...",0.060294,0.616631,NaN,NaN,NaN,NaN,0,NaN,36.0
4,5_abs_cos_sem,is_additive,cosine_abs + sem_sim,2,0.474747,-1.830071,NaN,NaN,0.818182,0.656566,...,"[0.888888888888889, 0.9090909090909091, 0.8686...",0.212179,0.715294,-4.715113,-0.817363,36.0,36.0,0,-0.400134,NaN
5,6_signed_cos_sem,is_additive,signed_cos + sem_sim,2,0.272727,NaN,NaN,-0.381207,0.666667,0.515152,...,"[0.7272727272727273, 0.7474747474747475, 0.757...",0.642336,0.709856,-0.633324,0.066089,35.0,36.0,0,-0.701016,NaN
6,7_multi_abs_sem,regime,cosine_abs + sem_sim,2,0.417182,NaN,NaN,NaN,0.713833,0.656369,...,"[0.8052278325123154, 0.7995748592556388, 0.753...",0.274773,0.683761,NaN,NaN,NaN,NaN,0,NaN,36.0


In [96]:
print("Total pairs:", len(df))
print("\nRegime counts:")
print(df['regime'].value_counts())
print("\nBinary balance:")
print(df['is_additive'].value_counts())
print("\nMissing values:")
print(df[['cosine_abs', 'sem_sim', 'is_additive', 'cos', 'regime']].isna().sum())
print("\ncos distribution:")
print(df['cos'].describe())

Total pairs: 36

Regime counts:
regime
mixed          19
emergent        6
dominant        5
additive        3
suppressive     3
Name: count, dtype: int64

Binary balance:
is_additive
0    33
1     3
Name: count, dtype: int64

Missing values:
cosine_abs     0
sem_sim        0
is_additive    0
cos            0
regime         0
dtype: int64

cos distribution:
count    36.000000
mean      0.161447
std       0.232398
min      -0.522500
25%       0.014825
50%       0.187750
75%       0.292325
max       0.694800
Name: cos, dtype: float64


In [97]:
print(df[df['is_additive'] == 1][['trait_a', 'trait_b', 'cos', 'cosine_abs', 'sem_sim', 'cosine_abs']].sort_values('cos'))
print("\nAdditive pairs — cos stats:")
print(df[df['is_additive'] == 1]['cos'].describe())
print("\nNon-additive pairs — cos stats:")
print(df[df['is_additive'] == 0]['cos'].describe())

      trait_a        trait_b     cos  cosine_abs   sem_sim  cosine_abs
6   apathetic  power_seeking  0.0063      0.0063  0.291267      0.0063
31   humorous  power_seeking  0.0727      0.0727  0.190971      0.0727
24  formality  power_seeking  0.1832      0.1832  0.235060      0.1832

Additive pairs — cos stats:
count    3.000000
mean     0.087400
std      0.089361
min      0.006300
25%      0.039500
50%      0.072700
75%      0.127950
max      0.183200
Name: cos, dtype: float64

Non-additive pairs — cos stats:
count    33.000000
mean      0.168179
std       0.240858
min      -0.522500
25%       0.015000
50%       0.226200
75%       0.302600
max       0.694800
Name: cos, dtype: float64


In [98]:
df['has_halluc'] = (df['trait_a']=='hallucinating') | (df['trait_b']=='hallucinating')
print(pd.crosstab(df['has_halluc'], df['is_additive']))


is_additive   0  1
has_halluc        
False        25  3
True          8  0
